In [3]:
import json
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

In [4]:
load_dotenv()
openai_client = OpenAI()

In [5]:
documents = load_faq_data()
index = build_index(documents)

If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [11]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [15]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [7]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

We register our `search` function along with the schema from earlier lessons:

In [12]:
agent_tools = Tools()
agent_tools.add_tool(search)

You can look at what ToyAIKit produced:

In [13]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.

## The chat interface and runner

Create the chat interface and a callback, then build the runner:

In [14]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)


In [16]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The `chat_interface` handles display in the notebook. The `callback`
renders model messages and tool calls as they happen. The runner runs
the agent loop, the same `while True` we wrote by hand. It sends
messages, executes function calls, adds tool outputs back, and repeats
until the model is done.

We pick `gpt-5.4-mini` here on purpose. Without it, ToyAIKit falls
back to a smaller, faster default that doesn't follow the instructions
as reliably.

## Running one prompt

Run a single prompt:

In [17]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


We used the typo "Olama" on purpose. The agent searches and gets poor
results, then retries with "Ollama". The recovery is the same as the
handwritten loop. The notebook output is nicer to watch. Each tool
call and message renders inline, so you can look at every search
result.


The `result` is a `LoopResult` with `all_messages` (the full
conversation), token counts, and `cost` (computed from token usage).

In [18]:
result = runner.loop(
    prompt="How do I run Ollama locally?",
    callback=callback,
)

-> Response received


-> Response received


## Cost and tokens

Look at what the call cost:

In [19]:
result.cost

CostInfo(input_cost=Decimal('0.00111975'), output_cost=Decimal('0.000927'), total_cost=Decimal('0.00204675'))

Useful while developing - especially with multi-turn agents where one
prompt can trigger several model calls. The handwritten loop made you
compute this by hand. The framework keeps a running total for you.

You can also look at the full message history.

In [20]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse only the facts from the CONTEXT when answering the QUESTION.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Ollama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"run Ollama locally"}', call_id='call_pSUM6miPHRMC3ZYFu00cYkGx', name='search', type='function_call', id='fc_012f313887f5e059006aaec625038c87d2b770cf69bfd33c15', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_pSUM6miPHRMC3ZYFu00cYkGx',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **ma

This is just a list - the same `messages` list we maintained by hand.

## Continuing the conversation

Take the messages from the previous result and pass them as
`previous_messages` on the next `loop` call:

In [21]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


The runner picks up where the last call left off, with the same agent
loop and an extended history. The model knows "different model" refers
to Ollama because it sees the previous turn in memory. Without that
history, it would have no idea what we're asking about.


## Interactive chat

For a chat-like workflow, run the built-in input loop:


In [22]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse only the facts from the CONTEXT when answering the QUESTION.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run a docker inside a docker', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"run docker inside a docker"}', call_id='call_xEx8FJSQPfMkBHZMStvJ8TDe', name='search', type='function_call', id='fc_010cd399253ffd5d006aaec8cc99e487d291207f44f31bd406', async_=None, caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_xEx8FJSQPfMkBHZMStvJ8TDe', 'output': '[\n  {\n    "id": "e8df9f0d12",\n    "course": "llm-zoomcamp",\n    "section": "Module 6: Best Practices",\n    "question": "Docker: When trying to run a streamlit app using docker-compose, I get: Error response from daemon: failed to create task for container: failed 

Type questions and get answers. Type "stop" to exit.